# Lokalizacja decyzji

![Obraz](https://hackmd.io/_uploads/SkjywwKoWl.png)

*Obraz wygenerowany za pomocą ChatGPT*

## Wstęp

Model potrafi rozpoznawać obiekty na obrazach, ale nie wiadomo, na co dokładnie patrzy, podejmując decyzję. Czasami faktycznie wykrywa właściwy obiekt, a czasami kieruje się przypadkowymi cechami, jak tło czy kolory. Dlatego chcemy sprawdzić, które fragmenty obrazu mają największy wpływ na jego odpowiedź.

## Zadanie

Otrzymujesz wytrenowany klasyfikator binarny. Twoim zadaniem jest wygenerowanie map istotności (heatmap) wskazujących regiony obrazu, które miały największy wpływ na jego decyzję.

## Dane

Zbiór danych stanowi podzbiór **COCO2017**, zawierający obrazy, na których zawsze występuje klasa rozpoznawana przez model jako pozytywna.

Jako dane otrzymasz jedynie **zbiór walidacyjny** – 1200 obrazów.

Wszystkie obrazy mają rozmiar **224 × 224 piksele**.

Otrzymujesz również **model ResNet-18**, wytrenowany do zadania **klasyfikacji binarnej**. Model rozpoznaje, czy na obrazie znajduje się jeden, konkretny obiekt.

Podczas testowania zostanie użyty ten sam model. Zbiór testowy skałada się z 1200 obrazów.

## Kryterium oceny

Twoje rozwiązanie będzie oceniane na podstawie jakości wygenerowanych map istotności.

Dla każdej heatmapy obliczana jest metryka **Intersection over Union (IoU)** pomiędzy:

* regionem wskazanym przez heatmapę (zbiór rozmyty),
* rzeczywistą lokalizacją obiektu zdefiniowaną na podstawie maski.

IoU definiujemy jako:

$$
IoU = \frac{|H \cap G|}{|H \cup G|}
$$

gdzie:

* $H$ – obszar istotny wskazany przez model,
* $G$ – obszar obiektu z anotacji COCO.

Ostateczna ocena będzie obliczana według wzoru:

$$results\_evaluation = 
\begin{cases} 
    0 &\quad \text{jeżeli IoU} \le 0.25  \\
    100 \cdot \dfrac{\text{IoU} - 0.191}{0.245 - 0.191} &\quad \text{jeżeli }  0.191 \leq \text{IoU\_{mean}} \le 0.245 \\
    100 &\quad \text{w pozostałych przypadkach}
\end{cases}$$

## Ograniczenia

* Rozwiązanie będzie uruchamiane **bez dostępu do internetu**
* Dostępne jest **GPU**
* Cała ewaluacja nie może trwać dłużej niż **3 minuty**
* Nie wolno modyfikować wag modeli
* Dozwolone biblioteki: `torch`, `torchvision`, `numpy`, `cv2`

Twoje rozwiązanie będzie testowane na Platformie Konkursowej bez dostępu do internetu oraz w środowisku z GPU.

##  Pliki Zgłoszeniowe
Ten notebook uzupełniony o Twoje rozwiązanie.


## Ewaluacja

Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`.

Za to zadanie możesz zdobyć pomiędzy 0 a 100 punktów. Liczba punktów, którą zdobędziesz, będzie wyliczona na (tajnym) zbiorze testowym na Platformie Konkursowej na podstawie wyżej wspomnianego wzoru, zaokrąglona do liczby całkowitej. Jeśli Twoje rozwiązanie nie będzie spełniało powyższych kryteriów lub nie będzie wykonywać się prawidłowo, otrzymasz za zadanie 0 punktów.


# Kod startowy

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

import os
import random
import numpy as np
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

FINAL_EVALUATION_MODE = False  # Podczas sprawdzania ustawimy tę flagę na True.


In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

def download_data():
    """Pobiera zbiór danych z Google Drive i zapisuje go w folderze 'data'."""
    import shutil
    import gdown

    # Utwórz lub zresetuj folder 'data'
    if not os.path.exists('data'):
        os.makedirs('data')
    else:
        shutil.rmtree('data')
        os.makedirs('data')

    # Pobierz plik z Google Drive i zapisz go w folderze 'data'
    url_model = "https://drive.google.com/file/d/1b6WhjG_GDihBNKLIXtFcaclJKkTWbZZk/view?usp=drive_link"
    gdown.download(url_model, f'data/model.pth', quiet=True, fuzzy=True)

    url_data = "https://drive.google.com/file/d/1hVn9m0KRuE-0x7zpPYcwkxDFMNoaM-U9/view?usp=drive_link"
    gdown.download(url_data, f'data/val_data.npz', quiet=True, fuzzy=True)


if not FINAL_EVALUATION_MODE:
    download_data()

## Definicja modelu

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# Definicja architektury modelu.
class ResNet18Classifier(nn.Module):
    def __init__(self, pretrained=False, num_classes=1):
        super().__init__()

        if pretrained:
            weights = models.ResNet18_Weights.DEFAULT
        else:
            weights = None

        resnet = models.resnet18(weights=weights)

        self.encoder = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.encoder(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.head(x)
        return x

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# Inicjalizacja modeli
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_A = ResNet18Classifier(num_classes=1).to(device)

# Wczytanie wytrenowanych wag
model_A.load_state_dict(torch.load("data/model.pth", map_location=device))
model_A.eval()

## Ładowanie danych

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

class CocoSegmentationDataset(Dataset):
    def __init__(self, npz_path, mode="val"):
        """
        npz_path: Ścieżka do pliku .npz (np. 'data/val_data.npz')
        mode: Klucz wewnątrz npz
        """
        if not os.path.exists(npz_path):
            raise FileNotFoundError(f"Nie znaleziono pliku: {npz_path}")

        # Wczytanie danych z NPZ
        data = np.load(npz_path, allow_pickle=True)
        
        # Mapowanie klucza
        key = f"{mode}_ds"
        if key not in data:
            raise KeyError(f"Plik {npz_path} nie zawiera klucza {key}")
            
        self.samples = data[key].tolist()
        
        # Definicja transformacji ImageNet
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Konwersja obrazu (uint8 [H,W,C] -> float32 [C,H,W])
        # 'image' to Twój JPG z oryginalnego datasetu
        img_np = sample["image"]
        img_tensor = torch.from_numpy(img_np).to(torch.float32) / 255.0
        img_tensor = img_tensor.permute(2, 0, 1) # Zmiana na [C, H, W]
        img_tensor = self.normalize(img_tensor)
        
        # 4. Konwersja maski (uint8 [H,W] -> float32 [1,H,W])
        # 'mask' to Twoja maska PNG
        mask_np = sample["mask"]
        mask_tensor = torch.tensor(mask_np / 255.0, dtype=torch.float32)
        
        # Dodanie wymiaru kanału jeśli maska jest 2D
        if mask_tensor.ndim == 2:
            mask_tensor = mask_tensor.unsqueeze(0)
            
        return img_tensor, mask_tensor

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# Inicjalizacja
val_dataset = CocoSegmentationDataset("data/val_data.npz")
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True)

## Twoje rozwiązanie

In [ ]:
# Twoje rozwiązanie - tylko tu wprowadzaj zmiany.
# Nie zmieniaj argumentów funkcji, ani ilości zwracanych wartości.
# Funkcja została uzupełniona przykładowym rozwiązaniem, które dalekie jest
# od optymalnego - w szczególności w ogóle nie wykorzystuje danych modeli.
# Ma służyć jako punkt wyjścia do budowania rozwiązania.

def your_solution(img, model):
    _, H, W = img.shape

    # Maska wygenerowana losowo.
    mask = torch.randint(0, 2, (H, W), dtype=torch.float32)

    return mask

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# Wyświetlenie przykładowego obrazu z datasetu.
if not FINAL_EVALUATION_MODE:
    import matplotlib.pyplot as plt

    def plot_example(dataset, model, your_solution):
        img, mask = dataset[0]
        pred = your_solution(img, model)
        if torch.is_tensor(pred):
            pred = pred.detach().cpu().numpy()
        mask_to_show = mask.squeeze().cpu().numpy()

        # Permutacja kanałów    
        img_display = img.permute(1, 2, 0).numpy()
        # Odwrócenie normalizacji dla obrazka
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        # Przycięcie wartości do zakresu [0, 1] dla poprawnej wizualizacji
        img_display = np.clip(img_display, 0, 1)

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.title("Oryginał")
        plt.imshow(img_display)
        
        plt.subplot(1, 3, 2)
        plt.title("Maska (GT)")
        plt.imshow(mask_to_show, cmap='gray')
        
        plt.subplot(1, 3, 3)
        plt.title("Predykcja")
        plt.imshow(pred, cmap='jet', alpha=0.5)
        plt.imshow(img_display, alpha=0.5)
        plt.show()

    # Uruchomienie funkcji wizualizującej przykładowy obraz z datasetu wraz z maską GT i predykcją.
    plot_example(val_dataset, model_A, your_solution)

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# Oblicza metrykę IoU między dwiema maskami.

def compute_iou(pred, target):
    # Upewniamy się, że tensory są binarne (0 lub 1)
    pred = (pred > 0.5).float()
    target = (target > 0.5).float()
    
    intersection = (pred * target).sum()
    union = (pred + target).clamp(0, 1).sum()
    return (intersection + 1e-8) / (union + 1e-8)

## Ewaluacja

Uruchomienie poniższej komórki pozwoli sprawdzić, ile punktów zdobyłoby Twoje rozwiązanie na danych treningowych. Przed wysłaniem upewnij się, że cały notebook wykonuje się od początku do końca bez błędów po ustawieniu flagi *FINAL_EVALUATION_MODE = True* i bez konieczności ingerencji użytkownika po wybraniu opcji "Run All".

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# Obliczanie wyniku dla uczestników (jawne, korzysta z val_loader)
def compute_score(loader, model):
    model.eval()
    ious = []
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for img, mask in loader:
            img, mask = img.to(device), mask.to(device)
            
            # Przekazujemy pojedynczy obraz (C, H, W) do rozwiązania uczestnika
            # Zakładamy, że your_solution zwraca maskę (H, W) lub (1, H, W)
            pred = your_solution(img[0], model)
            mask = mask.to(pred.device)
            
            # Dopasowanie kształtów do IoU
            ious.append(compute_iou(pred.squeeze(), mask.squeeze()))
    
    mIoU = sum(ious) / len(ious)
    
    # Skalowanie wyniku: 0.191 -> 0 pkt, 0.245 -> 100 pkt
    raw_score = 100 * (mIoU - 0.191) / (0.245 - 0.191)
    score = int(torch.clamp(raw_score, min=0, max=100).round().item())
    
    return mIoU, score

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# Ewaluuje Twoje rozwiązanie na zbiorze walidacyjnym obliczając średnie IoU.
# Podobną wartość powinno otrzymać na tajnym zbiorze testowym.

if not FINAL_EVALUATION_MODE:
    mIoU, score = compute_score(val_loader, model_A)
    print(f"mIoU: {mIoU:.4f} | Estymowana liczba punktów: {score}/100")